# Model Architecture Design

This notebook designs the classification architecture and establishes the training strategy for the skin lesion classification system.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB2
from pathlib import Path
import json

2026-03-04 16:27:09.616591: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-04 16:27:10.654769: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-04 16:27:23.443016: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
PROJECT_ROOT = Path("../").resolve()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed" / "classification"
ORGANIZED_DIR = DATA_DIR / "organized"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

# Create models directory
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Load configuration
with open(PROJECT_ROOT / "config.json", "r") as f:
    CONFIG = json.load(f)

## 1. Model Selection: EfficientNet

In [4]:
# Model Configuration
IMG_SIZE = 256
IMG_CHANNELS = 3
NUM_CLASSES = 8
BATCH_SIZE = 64

CLASS_NAMES = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']

print("Model Configuration:")
print(f"  Input shape: ({IMG_SIZE}, {IMG_SIZE}, {IMG_CHANNELS})")
print(f"  Number of classes: {NUM_CLASSES}")
print(f"  Classes: {CLASS_NAMES}")
print(f"  Batch size: {BATCH_SIZE}")

Model Configuration:
  Input shape: (256, 256, 3)
  Number of classes: 8
  Classes: ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']
  Batch size: 64


## 2. Transfer Learning

### Two-Phase Training Approach

**Phase 1: Feature Extraction**
- Freeze all EfficientNet base layers
- Train only the custom classification head
- Higher learning rate (1e-2)
- Allows the model to adapt to our dataset while preserving ImageNet features

**Phase 2: Fine-Tuning**
- Unfreeze the entire model
- Lower learning rate (1e-4)
- Fine-tune all layers to optimize for skin lesion features
- Prevents catastrophic forgetting of useful features

In [5]:
def create_efficientnet_b2_model(num_classes=8, img_size=256, dropout_rate=0.2):
    """
    Create EfficientNetB2 model with custom classification head.
    
    Architecture:
    - Input: 256x256x3 images
    - Data augmentation (rotation, translation, flip, contrast)
    - EfficientNetB2 base (pre-trained on ImageNet)
    - Global Average Pooling
    - Dropout for regularization
    - Dense layer with softmax output (8 classes)
    """
    
    # Data augmentation pipeline
    img_augmentation = keras.Sequential(
        [
            layers.RandomRotation(factor=0.15),
            layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
            layers.RandomFlip(),
            layers.RandomContrast(factor=0.1),
        ],
        name="img_augmentation",
    )
    
    # Input layer
    inputs = layers.Input(shape=(img_size, img_size, 3))
    
    # Apply augmentation
    x = img_augmentation(inputs)
    
    # EfficientNetB2 base (without top layers)
    base_model = EfficientNetB2(
        include_top=False,
        input_tensor=x,
        weights="imagenet"
    )
    
    # Global Average Pooling
    x = layers.GlobalAveragePooling2D(name="avg_pool")(base_model.output)
    
    # Dropout for regularization
    x = layers.Dropout(dropout_rate, name="top_dropout")(x)
    
    # Output layer
    outputs = layers.Dense(num_classes, activation="softmax", name="pred")(x)
    
    # Create model
    model = keras.Model(inputs, outputs, name="EfficientNetB2_SkinLesion")
    
    return model, base_model

# Create model
model, base_model = create_efficientnet_b2_model()
model.summary()

I0000 00:00:1772641678.699187  127385 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13685 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "EfficientNetB2_SkinLesion"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ img_augmentation    │ (None, 256, 256,  │          0 │ input_layer[0][0] │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 256, 256,  │          0 │ img_augmentation… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 256, 256,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 256, 256,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 257, 257,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 128, 128,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 128, 128,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 128, 128,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 128, 128,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 128, 128,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 128, 128,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 128, 128,  │          0 │ block1a_activati

 Total params: 7,779,841 (29.68 MB)

 Trainable params: 7,712,266 (29.42 MB)

 Non-trainable params: 67,575 (263.97 KB)

## 3. Training Configuration

### Loss Function
- **Categorical Cross-Entropy**: Standard for multi-class classification
- Will use sparse version with integer labels

### Optimizer
- **Adam**: Adaptive moment estimation
- Phase 1 LR: 1e-2 (feature extraction)
- Phase 2 LR: 1e-4 (fine-tuning)

### Callbacks
- **EarlyStopping**: Prevent overfitting
- **ModelCheckpoint**: Save best model
- **ReduceLROnPlateau**: Adaptive learning rate

In [6]:
def get_training_config(phase=1):
    """
    Get training configuration for specific phase.
    
    Args:
        phase: 1 for feature extraction, 2 for fine-tuning
    """
    if phase == 1:
        return {
            'learning_rate': 1e-2,
            'epochs': 20,
            'freeze_base': True,
            'description': 'Feature extraction - training classification head only'
        }
    else:
        return {
            'learning_rate': 1e-4,
            'epochs': 30,
            'freeze_base': False,
            'description': 'Fine-tuning - training all layers'
        }

def get_callbacks(model_name, results_dir):
    """
    Create training callbacks.
    """
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=str(results_dir / f'{model_name}_best.h5'),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        )
    ]
    return callbacks

# Display configurations
print("Training Strategy:")
print("\nPhase 1:")
for k, v in get_training_config(1).items():
    print(f"  {k}: {v}")
print("\nPhase 2:")
for k, v in get_training_config(2).items():
    print(f"  {k}: {v}")

Training Strategy:

Phase 1:
  learning_rate: 0.01
  epochs: 20
  freeze_base: True
  description: Feature extraction - training classification head only

Phase 2:
  learning_rate: 0.0001
  epochs: 30
  freeze_base: False
  description: Fine-tuning - training all layers


## 4. Data Loading Strategy

In [7]:
def load_data_from_directory(data_dir, batch_size=64, validation_split=0.2, seed=42):
    """
    Load training and validation datasets from organized directory.
    
    Expected structure:
    data_dir/
      class1/
        image1.jpg
        image2.jpg
      class2/
        ...
    """
    
    train_ds = keras.preprocessing.image_dataset_from_directory(
        data_dir,
        validation_split=validation_split,
        subset="training",
        seed=seed,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        label_mode='int'
    )
    
    val_ds = keras.preprocessing.image_dataset_from_directory(
        data_dir,
        validation_split=validation_split,
        subset="validation",
        seed=seed,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        label_mode='int'
    )

    class_names = train_ds.class_names

    
    # Performance optimization
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
    
    return train_ds, val_ds, class_names

In [8]:
# Test data loading
print("Loading data from directory...")
train_dir = ORGANIZED_DIR / "train"

train_ds, val_ds, class_names = load_data_from_directory(
    train_dir,
    batch_size=BATCH_SIZE,
    validation_split=0.2
)
print(f"\nTraining batches: {len(train_ds)}")
print(f"Validation batches: {len(val_ds)}")
print(f"\nClass names: {class_names}")

Loading data from directory...
Found 7010 files belonging to 7 classes.
Using 5608 files for training.
Found 7010 files belonging to 7 classes.
Using 1402 files for validation.

Training batches: 88
Validation batches: 22

Class names: ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']


## 5. Evaluation Metrics Strategy

Given the severe class imbalance, we prioritize metrics beyond simple accuracy:

### Primary Metrics
- **Accuracy**: Overall correctness (can be misleading with imbalance)
- **Precision**: TP / (TP + FP) - reliability of positive predictions
- **Recall**: TP / (TP + FN) - ability to find all positive cases
- **F1-Score**: Harmonic mean of precision and recall

### Class-Specific Focus
- **Minority class recall**: Critical for DF, VASC, SCC (rare but important)
- **Malignant class performance**: MEL, BCC, SCC detection priority

### Visualization
- Confusion matrix
- Per-class metrics
- ROC curves (for binary malignant vs benign)

In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

def evaluate_model(model, test_ds, class_names):
    """
    Comprehensive model evaluation.
    """
    # Get predictions
    y_true = []
    y_pred = []
    
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        y_pred.extend(np.argmax(preds, axis=1))
        y_true.extend(labels.numpy())
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro'),
        'recall_macro': recall_score(y_true, y_pred, average='macro'),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted'),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted'),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted')
    }
    
    return metrics, y_true, y_pred

print("\nMetrics to track:")
print("  - Overall Accuracy")
print("  - Macro Precision/Recall/F1 (equal weight to all classes)")
print("  - Weighted Precision/Recall/F1 (accounts for class frequency)")
print("  - Per-class metrics")


Metrics to track:
  - Overall Accuracy
  - Macro Precision/Recall/F1 (equal weight to all classes)
  - Weighted Precision/Recall/F1 (accounts for class frequency)
  - Per-class metrics
